In [ ]:
!pip install --upgrade --quiet \
    langchain-community \
    langchain-openai \
    chromadb \
    pypdf \
    pandas \
    streamlit \
    python-dotenv


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0

In [ ]:
import os
import json
import uuid
from openai import OpenAI
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import bs4


In [ ]:



# =============================
# 1. Setup
# =============================
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get("OPENROUTER_API_KEY" ),
)



In [ ]:
def load_data(source):
    if source.startswith("http" ):
        print(f"🌐 جاري تحميل الرابط...")
        loader = WebBaseLoader(
            web_path=(source,),
            bs_kwargs=dict(parse_only=bs4.SoupStrainer(name=("article", "h1", "h2", "h3", "p")))
        )
    else:
        print(f"📄 جاري تحميل ملف PDF...")
        loader = PyPDFLoader(source)
    return loader.load()


In [ ]:
import hashlib
import os

# 1. دالة الـ Embeddings (كما هي)
def get_embedding_func():
    return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. دالة إنشاء الـ Vectorstore (المعدلة لتكون أكثر استقراراً)
def create_vectorstore(chunks, embedding_func, vectorstore_path):
    print(f"🛠️ جاري إنشاء قاعدة بيانات في المجلد: {vectorstore_path}")

    # تنظيف البيانات ومنع التكرار
    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, doc.page_content)) for doc in chunks]
    unique_ids, unique_chunks = set(), []
    for chunk, id in zip(chunks, ids):
        if id not in unique_ids:
            unique_ids.add(id)
            unique_chunks.append(chunk)

    # إنشاء قاعدة البيانات (تم حذف .persist() لتجنب أخطاء الـ Readonly في Colab)
    vectorstore = Chroma.from_documents(
        documents=unique_chunks,
        ids=list(unique_ids),
        embedding=embedding_func,
        persist_directory=vectorstore_path
    )

    return vectorstore


In [ ]:
source_input = ""
# --- الكود السحري لتوليد اسم مجلد فريد بناءً على اسم الملف ---
# نأخذ اسم الملف فقط ونحوله لـ "بصمة" (Hash) قصيرة
file_name = os.path.basename(source_input)
folder_suffix = hashlib.md5(file_name.encode()).hexdigest()[:8]
unique_path = f"vectorstore_{folder_suffix}"

# الآن نستدعي الدوال باستخدام الاسم الفريد
embedding_func = get_embedding_func()
pages = load_data(source_input)

# إنشاء أو تحميل قاعدة البيانات الفريدة
if not os.path.exists(unique_path):
    vectorstore = create_vectorstore(pages, embedding_func, unique_path)
else:
    print(f"♻️ تم العثور على ذاكرة سابقة لهذا الملف في: {unique_path}")
    vectorstore = Chroma(persist_directory=unique_path, embedding_function=embedding_func)


/tmp/ipykernel_14632/27069682.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🌐 جاري تحميل الرابط...
🛠️ جاري إنشاء قاعدة بيانات في المجلد: vectorstore_dafbb3cd


In [ ]:

user_language_choice = ""
def run_smart_analysis(vectorstore,target_lang):
    retriever = vectorstore.as_retriever(search_kwargs={"k": 15})
    relevant_chunks = retriever.invoke("استخرج أهم الأفكار والنتائج الرئيسية للبودكاست")
    context = "\n\n".join([chunk.page_content for chunk in relevant_chunks])

    PROMPT_TEMPLATE = """
    You are a Content Analyzer Agent. Your goal is to prepare material for a professional podcast.
    Analyze the context and extract up to 15 unique, high-value topics.

    Content: {context}

    INSTRUCTION FOR SHORT CONTENT:
    - Assess the depth of the provided 'Context'.
    - If the context is rich and long, extract 15-20 unique topics. Set "content_status" to "sufficient".
    - If the context is short or lacks depth, extract only 5 high-quality topics. Set "content_status" to "limited" and provide a brief warning message in "system_note" in {target_lang}.

    Rules:
    - LANGUAGE RULE: Analyze the provided content and provide the output in {target_lang}.
    - STRICT PROHIBITION: Do NOT use any Chinese characters (中文), even if the model defaults to it.
    - Extract 15 to 20 distinct topics.
    - Focus on facts, stories, and controversial points suitable for a 15-minute dialogue.
    - The entire output content MUST be in {target_lang}.
    - Return ONLY valid JSON.

    Format:
    {{
      "content_status": "sufficient" or "limited",
      "system_note": "A message to the user if content is limited",
      "topics": [
        {{
          "title": "Topic Title",
          "key_points": ["Point 1", "Point 2"],
          "insight": "Deep takeaway",
          "discussion_angles": ["Question for the host", "Counter-argument"]
        }}
      ]
    }}
    """

    completion = client.chat.completions.create(
         #model="qwen/qwen-2.5-72b-instruct",
         model ="meta-llama/llama-3.3-70b-instruct",
         messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(context=context, target_lang=target_lang)}]

        # أزلنا response_format مؤقتاً لزيادة التوافق مع جميع الموديلات
    )


    raw_content = completion.choices[0].message.content.strip()

    # محاولة تنظيف النص إذا أضاف الموديل أي كلام خارج الـ JSON
    if "```json" in raw_content:
        raw_content = raw_content.split("```json")[1].split("```")[0].strip()
    elif "```" in raw_content:
        raw_content = raw_content.split("```")[1].split("```")[0].strip()

    try:
        return json.loads(raw_content)
    except json.JSONDecodeError:
        print("⚠️ فشل في تحويل النص إلى JSON، النص الخام المستلم:")
        print(raw_content)
        raise

In [ ]:
try:
    print("🧠 جاري تحليل المحتوى الآن... قد يستغرق ذلك بضع ثوانٍ.")
    final_analysis = run_smart_analysis(vectorstore, user_language_choice)

    print(f"✅ تم التحليل بنجاح! تم استخراج {len(final_analysis_json['topics'])} موضوعاً.")

    # طباعة النتيجة النهائية بشكل جميل ومنظم
    print("-" * 30)
    print(json.dumps(final_analysis_json, indent=2, ensure_ascii=False))
    print("-" * 30)

except Exception as e:
    print(f"❌ حدث خطأ أثناء التحليل: {e}")

🧠 جاري تحليل المحتوى الآن... قد يستغرق ذلك بضع ثوانٍ.
✅ تم التحليل بنجاح! تم استخراج 15 موضوعاً.
------------------------------
{
  "topics": [
    {
      "title": "Introduction to Assembly Language",
      "key_points": [
        "Basic understanding of hardware components",
        "Importance of assembly language in modern computing"
      ],
      "insight": "Assembly language provides a low-level interface to computer hardware, essential for optimizing performance and understanding system architecture.",
      "discussion_angles": [
        "What are the practical applications of assembly language in today's software development?",
        "How does assembly language compare to high-level languages in terms of performance and ease of use?"
      ]
    },
    {
      "title": "Computer Architecture Fundamentals",
      "key_points": [
        "ALU, CU, Registers",
        "Buses (Address, Data, Control)"
      ],
      "insight": "A deep understanding of computer architecture is c

In [ ]:
try:
    print("🧠 جاري تحليل المحتوى الآن... قد يستغرق ذلك بضع ثوانٍ.")
    final_analysis = run_smart_analysis(vectorstore, user_language_choice)

    print(f"✅ تم التحليل بنجاح! تم استخراج {len(final_analysis['topics'])} موضوعاً.")

    # طباعة النتيجة النهائية بشكل جميل ومنظم
    print("-" * 30)
    print(json.dumps(final_analysis, indent=2, ensure_ascii=False))
    print("-" * 30)

except Exception as e:
    print(f"❌ حدث خطأ أثناء التحليل: {e}")

🧠 جاري تحليل المحتوى الآن... قد يستغرق ذلك بضع ثوانٍ.
✅ تم التحليل بنجاح! تم استخراج 15 موضوعاً.
------------------------------
{
  "content_status": "sufficient",
  "topics": [
    {
      "title": "الأداء المتواضع لتشيلسي في الدوري الإنجليزي",
      "key_points": [
        "خسارة تشيلسي أمام نوتنجهام فورست بنتيجة 3-1",
        "استقبال هدف مبكر عقد مهمة الفريق"
      ],
      "insight": "الأداء غير المستقر لتشيلسي يمكن أن يؤدي إلى فقدان مركز مؤهل لدوري أبطال أوروبا.",
      "discussion_angles": [
        "ما هي أسباب التراجع المستمر في أداء تشيلسي؟",
        "كيف يمكن للنادي مواجهة هذه التحديات؟"
      ]
    },
    {
      "title": "مسؤولية اللاعبين في أخطاء الفريق",
      "key_points": [
        "جواو بيدرو يحمّل اللاعبين المسؤولية",
        "ضرورة مراجعة الأخطاء والعمل على تصحيحها"
      ],
      "insight": "مسؤولية اللاعبين تُظهر حاجة الفريق إلى تحليل ذاتي شامل وتطبيق تغييرات فنية.",
      "discussion_angles": [
        "هل يتحمل المدرب مسؤولية أكبر في هذه الأخطاء؟",
        "كيف 

In [ ]:
try:
    print("🧠 جاري تحليل المحتوى الآن... قد يستغرق ذلك بضع ثوانٍ.")
    final_analysis = run_smart_analysis(vectorstore, user_language_choice)

    print(f"✅ تم التحليل بنجاح! تم استخراج {len(final_analysis['topics'])} موضوعاً.")

    # طباعة النتيجة النهائية بشكل جميل ومنظم
    print("-" * 30)
    print(json.dumps(final_analysis, indent=2, ensure_ascii=False))
    print("-" * 30)

except Exception as e:
    print(f"❌ حدث خطأ أثناء التحليل: {e}")

🧠 جاري تحليل المحتوى الآن... قد يستغرق ذلك بضع ثوانٍ.
✅ تم التحليل بنجاح! تم استخراج 16 موضوعاً.
------------------------------
{
  "content_status": "sufficient",
  "system_note": "",
  "topics": [
    {
      "title": "تحلیل نتيجة مباراة تشيلسي ونوتنجهام فورست",
      "key_points": [
        "نتيجة المباراة",
        "أهداف الفريقين",
        "Performances اللاعبين"
      ],
      "insight": "التحلیل العميق لنتيجة المباراة وتأثيرها على ترتيب الفريقين",
      "discussion_angles": [
        "ما هي الأسباب الرئيسية وراء خسارة تشيلسي؟",
        "كيف يمكن للفريق تحسين أدائه في المباريات القادمة؟"
      ]
    },
    {
      "title": "تأثير خسارة تشيلسي على فرص التأهل لدوري الأبطال",
      "key_points": [
        "ترتيب تشيلسي في الدوري",
        "فرص التأهل لدوري الأبطال",
        " hậu果 خسارة المباراة"
      ],
      "insight": "التحلیل العميق لخسارة تشيلسي وتأثيرها على فرص التأهل لدوري الأبطال",
      "discussion_angles": [
        "ما هي الحلول الممكنة لتحسين فرص التأهل؟",
        "كيف 